# Predicting Airline Satisfaction
### 01 · Análisis exploratorio de datos (EDA)

**Competencia:** [Kaggle · Playground Series S6E10](https://www.kaggle.com/competitions/playground-series-s6e10/overview)  
**Problema:** predecir si un pasajero está satisfecho con su experiencia de vuelo.  
**Variable objetivo:** `satisfaction` (`True`: satisfecho; `False`: insatisfecho).  
**Métrica del proyecto:** ROC-AUC, evaluada en la etapa de modelado.

**Objetivo de este notebook.** Conocer la estructura y calidad de los datos, describir cómo se distribuye la satisfacción e investigar posibles asociaciones entre las variables. Este análisis es **descriptivo**: no permite establecer relaciones causales ni entrena modelos.

**Panorama inicial:** el archivo de entrenamiento contiene **699.635 registros y 23 columnas**. El **44,36 %** de los pasajeros figura como satisfecho y encontramos **292 valores faltantes** en `Arrival Delay in Minutes`. Los detalles y las comprobaciones se presentan en las secciones siguientes.


## Guía de lectura

1. **Carga y estructura:** dimensiones, columnas y tipos de datos.
2. **Calidad y variable objetivo:** faltantes, duplicados y proporción de satisfacción.
3. **Variables categóricas:** clase, motivo del viaje y tipo de cliente.
4. **Estadística descriptiva:** comportamiento de las variables numéricas.
5. **Valoración de servicios:** satisfacción según las puntuaciones de cada servicio.
6. **Correlaciones:** asociaciones de Spearman con la satisfacción.
7. **Variables numéricas:** tasas de satisfacción por intervalos y gráficos de dispersión.
8. **Conclusiones:** principales hallazgos y limitaciones.

Las tablas y los gráficos se generan desde `train.csv`. Algunas visualizaciones también se guardan en la carpeta `images/` para documentar el proyecto en GitHub.


## 1. Carga de datos y estructura

Utilizamos `train.csv`, ubicado en la carpeta `data/`. La ruta se construye a partir de la ubicación del proyecto para evitar referencias a directorios personales y facilitar la ejecución desde **`notebooks/`** o desde la raíz del repositorio.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Buscar la carpeta raíz del proyecto.
carpeta_actual = Path.cwd()
if (carpeta_actual / "data" / "train.csv").is_file():
    proyecto = carpeta_actual
elif (carpeta_actual.parent / "data" / "train.csv").is_file():
    proyecto = carpeta_actual.parent
else:
    raise FileNotFoundError(
        "No se encontró data/train.csv. Verifica que descargaste los datos "
        "de Kaggle y que estás ejecutando el notebook desde el proyecto."
    )

ruta_train = proyecto / "data" / "train.csv"
carpeta_imagenes = proyecto / "images"
carpeta_imagenes.mkdir(exist_ok=True)


In [12]:
train = pd.read_csv(ruta_train)
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


### Dimensiones y tipos de datos

- Tenemos **699.635 filas** y **23 columnas**, incluidas `id` y `satisfaction`.
- Al excluir el identificador y el objetivo, quedan **21 características** disponibles para el modelado.
- Las calificaciones de algunos servicios se almacenan como números enteros, pero representan **escalas ordinales**; su interpretación requiere cuidado.
- `id` identifica registros y será útil para la entrega a Kaggle, pero **no** se utilizará como característica del modelo.


In [21]:
print("DIMENSIONES:")
print(train.shape)

print("\nCOLUMNAS:")
print(train.columns.tolist())

print("\nINFORMACIÓN GENERAL:")
train.info()

DIMENSIONES:
(699635, 23)

COLUMNAS:
['id', 'Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']

INFORMACIÓN GENERAL:
<class 'pandas.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departu

## 2. Calidad de datos y distribución del objetivo

Comprobamos valores faltantes, identificadores, registros duplicados y categorías. En este notebook **no imputamos valores ni eliminamos observaciones**: esas decisiones corresponden al preprocesamiento.


In [23]:
train.isna().sum()

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64

**Valores faltantes:** `Arrival Delay in Minutes` es la única columna con ausencias: **292 registros** de 699.635 (**0,0417 %**). Más adelante imputamos estos valores con un procedimiento ajustado únicamente sobre el conjunto de entrenamiento, para evitar utilizar información de validación.

El siguiente diagnóstico revisa también los identificadores, los duplicados sin considerar `id` y la composición de las variables categóricas.


In [ ]:
print("ID único:", train["id"].is_unique)
print("Duplicados completos (sin id):", train.drop(columns="id").duplicated().sum())

for columna in ["Gender", "Customer Type", "Type of Travel", "Class"]:
    print(f"\n{columna}:")
    print(train[columna].value_counts(dropna=False))

### Distribución de la variable objetivo

En `train.csv`, el **55,64 %** de los pasajeros aparece como insatisfecho y el **44,36 %** como satisfecho. Existe una diferencia moderada en la proporción de clases, pero no un desbalance extremo. Utilizaremos una partición estratificada durante el modelado para conservar estas proporciones en entrenamiento y validación.


In [ ]:
# Porcentaje de pasajeros en cada clase.
(train["satisfaction"].value_counts(normalize=True) * 100).round(2)


## 3. Variables categóricas: ¿quién está más satisfecho?

### Clase de viaje

Comparamos la **tasa de satisfacción dentro de cada clase** y el número de pasajeros que la integra. Es importante distinguir este porcentaje de la participación de cada clase en el total de registros.


In [46]:
analisis_clase = train.groupby("Class").agg(
    pasajeros=("satisfaction", "size"),
    satisfechos=("satisfaction", "sum"),
    tasa_satisfaccion=("satisfaction", "mean")
)

analisis_clase["tasa_satisfaccion"] *= 100

analisis_clase.round(2)

,pasajeros,satisfechos,tasa_satisfaccion
Class,,,
Business,342212,248210,72.53
Eco,327404,54859,16.76
Eco Plus,30019,7270,24.22


In [ ]:
# Reutilizar el resumen anterior, sin volver a calcular los porcentajes.
tasas_clase = analisis_clase["tasa_satisfaccion"].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.bar(tasas_clase.index, tasas_clase.values, width=0.6, color="#287D8E")
ax.bar_label(barras, fmt="%.1f%%", padding=4)

ax.set_title("Satisfacción según la clase de viaje", fontsize=14, weight="bold")
ax.set_xlabel("Clase de viaje")
ax.set_ylabel("Pasajeros satisfechos (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", linestyle="--", alpha=0.2)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


**Interpretación:** la satisfacción es mayor entre pasajeros de **Business (72,53 %)**, seguida de **Eco Plus (24,22 %)** y **Eco (16,76 %)**. Esta comparación es descriptiva: no demuestra que cambiar de clase sea la causa de una mayor satisfacción.


### Motivo del viaje

Aquí comparamos el porcentaje de pasajeros satisfechos en viajes de negocios y viajes personales.

In [48]:
analisis_viaje = train.groupby("Type of Travel").agg(
    pasajeros=("satisfaction", "size"),
    satisfechos=("satisfaction", "sum"),
    tasa_satisfaccion=("satisfaction", "mean")
)

analisis_viaje["tasa_satisfaccion"] *= 100

analisis_viaje.round(2)

,pasajeros,satisfechos,tasa_satisfaccion
Type of Travel,,,
Business travel,497441,290673,58.43
Personal Travel,202194,19666,9.73


El porcentaje de satisfacción es **58,43 %** entre quienes viajan por negocios y **9,73 %** en viajes personales. Para entender esta diferencia es necesario observar su combinación con la clase de viaje.

### Clase y motivo del viaje

La primera tabla muestra **número de pasajeros** en cada combinación; la segunda presenta la **tasa de satisfacción dentro de esa combinación**. Son medidas distintas y no deben confundirse.


In [51]:
tabla_cruzada = pd.crosstab(
    train["Type of Travel"],
    train["Class"]
)

tabla_cruzada

Class,Business,Eco,Eco Plus
Type of Travel,,,
Business travel,338272,141612,17557
Personal Travel,3940,185792,12462


In [ ]:
# Porcentaje de satisfacción para cada combinación de clase y motivo de viaje.
satisfaccion_cruzada = pd.crosstab(
    index=train["Type of Travel"],
    columns=train["Class"],
    values=train["satisfaction"],
    aggfunc="mean"
) * 100

satisfaccion_cruzada.round(2)


In [ ]:
# Graficar las tasas de la tabla anterior sin recalcular el agregado.
tasas_clase_viaje = satisfaccion_cruzada.T.reindex(["Business", "Eco Plus", "Eco"])

fig, ax = plt.subplots(figsize=(10, 5))
tasas_clase_viaje.plot(
    kind="bar", ax=ax, width=0.7, color=["#287D8E", "#E6A04B"]
)
ax.set_title("Satisfacción según clase y motivo del viaje", fontsize=14, weight="bold")
ax.set_xlabel("Clase de viaje")
ax.set_ylabel("Pasajeros satisfechos (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Motivo del viaje")
ax.grid(axis="y", linestyle="--", alpha=0.2)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

for barras in ax.containers:
    ax.bar_label(barras, fmt="%.1f%%", padding=3)

plt.tight_layout()
fig.savefig(carpeta_imagenes / "satisfaction_class_travel.png", dpi=220, bbox_inches="tight")
plt.show()


**Lectura conjunta:** entre los viajes de negocios, la satisfacción aumenta desde **26,00 % en Eco** hasta **73,25 % en Business**. En los viajes personales, las tasas de las tres clases están aproximadamente entre **9,54 % y 11,19 %**. Esto muestra que la relación entre clase y satisfacción puede variar según el motivo de viaje.

Esta observación no permite establecer causalidad ni demuestra, por sí sola, una mejora predictiva. La combinación *Personal Travel + Business* contiene **3.940 pasajeros**, por lo que también conviene observar el tamaño de los grupos.


### Otras variables categóricas

Revisamos `Gender` y `Customer Type`. En el primer caso, las tasas de satisfacción son parecidas: **43,80 %** para `Female` y **44,91 %** para `Male`. El segundo resumen permite comparar la satisfacción y el tamaño de cada tipo de cliente.


In [ ]:
# Tasas en porcentaje para mantener el mismo criterio de las demás tablas.
resumen_genero = train.groupby("Gender")["satisfaction"].agg(
    pasajeros="size", tasa_satisfaccion="mean"
)
resumen_genero["tasa_satisfaccion"] *= 100
resumen_genero.round(2)


In [ ]:
resumen_cliente = train.groupby("Customer Type")["satisfaction"].agg(
    pasajeros="size", tasa_satisfaccion="mean"
)
resumen_cliente["tasa_satisfaccion"] *= 100
resumen_cliente.round(2)


## 4. Estadística descriptiva

Con `describe()` revisamos medidas como media, desviación estándar, cuartiles y valores extremos. Después comparamos las cuatro variables numéricas principales entre pasajeros satisfechos e insatisfechos.


In [59]:
train.describe()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes
count,699635.00000,699635.000000,699635.000000,699635.000000,699635.00000,699635.00000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699635.000000,699343.000000
mean,349817.00000,39.001928,1353.778702,2.778722,3.15621,2.81324,3.013734,3.265644,3.364156,3.568950,3.453910,3.517404,3.472847,3.805443,3.420291,3.380401,1.176209,1.134280
std,201967.37213,13.800943,954.422566,1.280419,1.47539,1.32955,1.254014,1.300202,1.288538,1.275333,1.308157,1.238474,1.271755,1.079175,1.210604,1.277870,5.982187,5.749304
min,0.00000,7.000000,67.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,174908.50000,27.000000,631.000000,2.000000,2.00000,2.00000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000,2.000000,3.000000,3.000000,2.000000,0.000000,0.000000
50%,349817.00000,40.000000,954.000000,3.000000,3.00000,3.00000,3.000000,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,0.000000,0.000000
75%,524725.50000,50.000000,1814.000000,4.000000,4.00000,4.00000,4.000000,4.000000,4.000000,5.000000,5.000000,4.000000,5.000000,5.000000,4.000000,4.000000,0.000000,0.000000
max,699634.00000,85.000000,4983.000000,5.000000,5.00000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,489.000000,491.000000


In [60]:
variables_numericas = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]
train.groupby("satisfaction")[variables_numericas].describe()

Age                                                     \
                 count       mean        std  min   25%   50%   75%   max   
satisfaction                                                                
False         389296.0  36.504131  14.858662  7.0  25.0  35.0  48.0  85.0   
True          310339.0  42.135220  11.610186  7.0  34.0  43.0  51.0  85.0   

             Flight Distance               ... Departure Delay in Minutes  \
                       count         mean  ...                        75%   
satisfaction                               ...                              
False               389296.0  1038.448556  ...                        0.0   
True                310339.0  1749.335704  ...                        0.0   

                    Arrival Delay in Minutes                                \
                max                    count      mean       std  min  25%   
satisfaction                                                                 
False         472.0                 389110.0  1.399383  6.341288  0.0  0.0   
True          489.0                 310233.0  0.801775  4.886579  0.0  0.0   

                               
              50%  75%    max  
satisfaction                   
False         0.0  0.0  446.0  
True          0.0  0.0  491.0  

[2 rows x 32 columns]

**Precaución:** aunque muchas valoraciones de servicios se registran con números del 0 al 5, son escalas **ordinales**. El valor `0` presenta comportamientos particulares en algunos servicios y no deberíamos asumir su significado sin verificar la documentación de los datos. Por eso no lo transformamos ni eliminamos durante el EDA.


## 5. Valoraciones de servicios y satisfacción

Cada casilla del mapa de calor corresponde a la **proporción de pasajeros satisfechos** entre quienes otorgaron una determinada calificación al servicio. Las combinaciones con **menos de 30 observaciones** se ocultan para no destacar tasas muy inestables; la ausencia de color **no equivale a 0 %**.

In [ ]:
# Tamaño del grupo y tasa de satisfacción según la calificación de Online boarding.
resumen_online = train.groupby("Online boarding")["satisfaction"].agg(
    pasajeros="size", tasa_satisfaccion="mean"
)
resumen_online["tasa_satisfaccion"] *= 100
resumen_online.round(2)


In [ ]:
# Servicios que vamos a comparar mediante tasas de satisfacción.
servicios = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]


In [ ]:
# Calcular porcentajes y número de pasajeros para cada valoración.
tasas = {}
cantidades = {}

for servicio in servicios:
    resumen = train.groupby(servicio)["satisfaction"].agg(["mean", "size"])
    tasas[servicio] = resumen["mean"] * 100
    cantidades[servicio] = resumen["size"]

tasas = pd.DataFrame(tasas).T.sort_index(axis=1)
cantidades = pd.DataFrame(cantidades).T.reindex(columns=tasas.columns)

tasas.round(1)


In [ ]:
# Enmascarar combinaciones sin datos o con menos de 30 pasajeros.
mascara = (cantidades < 30) | tasas.isna()

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(
    tasas,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    mask=mascara,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Pasajeros satisfechos (%)"},
    ax=ax
)

ax.set_title("Satisfacción según la valoración del servicio", fontsize=14, weight="bold", pad=16)
ax.set_xlabel("Calificación del servicio")
ax.set_ylabel("Servicio evaluado")
plt.tight_layout()

# Guardar antes de mostrar, para que el archivo generado coincida con la figura.
fig.savefig(carpeta_imagenes / "service_satisfaction_heatmap.png", dpi=240, bbox_inches="tight")
plt.show()


**Lectura del heatmap:** `Online boarding` presenta un cambio especialmente marcado entre las valoraciones 3 y 4; `Inflight entertainment` y `Seat comfort` también muestran diferencias importantes. Algunas categorías `0` no siguen el patrón de 1–5 y merecen revisión semántica antes de codificarlas.

> El mapa compara grupos de forma **univariada**. No permite afirmar que un servicio cause satisfacción ni establecer la importancia final de una característica en un modelo multivariable.

## 6. Correlaciones entre variables y con la satisfacción

Utilizamos el coeficiente de **Spearman** para explorar asociaciones **monotónicas**, dado que muchas puntuaciones de servicios son ordinales. Excluimos `id` y convertimos `satisfaction` a 0/1 para incluirla en el cálculo. Las variables categóricas de texto se estudiaron por separado.

Una correlación cercana a cero no descarta relaciones no lineales. Además, **correlación no implica causalidad**.


In [ ]:
# Seleccionar variables numéricas, omitiendo el identificador.
datos_corr = train.select_dtypes(include="number").drop(columns=["id"]).copy()
datos_corr["satisfaction"] = train["satisfaction"].astype(int)

# Correlaciones de Spearman.
correlaciones = datos_corr.corr(method="spearman")

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    correlaciones,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    cbar_kws={"label": "Correlación de Spearman"},
    ax=ax
)
ax.set_title("Correlaciones entre variables numéricas y satisfacción", fontsize=14, weight="bold", pad=14)
plt.tight_layout()
fig.savefig(carpeta_imagenes / "spearman_correlations.png", dpi=220, bbox_inches="tight")
plt.show()


In [71]:
correlacion_target = (
    correlaciones["satisfaction"]
    .drop("satisfaction")
    .sort_values(ascending=False)
)

correlacion_target.round(3)

Online boarding                      0.602
Inflight entertainment               0.433
Seat comfort                         0.413
On-board service                     0.354
Flight Distance                      0.337
Cleanliness                          0.335
Leg room service                     0.332
Inflight wifi service                0.287
Baggage handling                     0.280
Checkin service                      0.241
Age                                  0.214
Food and drink                       0.213
Ease of Online booking               0.192
Gate location                        0.024
Departure Delay in Minutes          -0.029
Departure/Arrival time convenient   -0.047
Arrival Delay in Minutes            -0.058
Name: satisfaction, dtype: float64

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
correlacion_target.sort_values().plot(kind="barh", ax=ax, color="#287D8E")
ax.set_title("Correlación de las variables con la satisfacción", fontsize=14, weight="bold", pad=14)
ax.set_xlabel("Coeficiente de Spearman")
ax.set_ylabel("")
ax.axvline(0, color="black", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(carpeta_imagenes / "spearman_target.png", dpi=220, bbox_inches="tight")
plt.show()


**Principales asociaciones con `satisfaction` (Spearman):** `Online boarding` (**0,602**), `Inflight entertainment` (**0,433**) y `Seat comfort` (**0,413**). Los retrasos presentan correlaciones negativas pequeñas: salida **−0,029** y llegada **−0,058**.

Algunas valoraciones de servicios también están relacionadas entre sí, lo que puede indicar información compartida. No eliminamos características en esta etapa: una correlación descriptiva **no es equivalente a la importancia que un modelo predictivo asignará a esa variable**.


## 7. Variables numéricas frente a la satisfacción

### 7.1. Porcentaje de satisfacción por intervalos

Agrupamos edad, distancia y retrasos en tramos para comparar la **proporción observada de pasajeros satisfechos**. Cada etiqueta indica también cuántos pasajeros hay en el grupo. No se trata de una regresión ni de una estimación causal.


In [ ]:
# Definir intervalos de lectura para cuatro características numéricas.
variables_intervalos = {
    "Age": {
        "bins": [-np.inf, 20, 30, 40, 50, 60, 70, np.inf],
        "labels": ["≤20", "21–30", "31–40", "41–50", "51–60", "61–70", ">70"],
        "titulo": "Edad"
    },
    "Flight Distance": {
        "bins": [-np.inf, 500, 1000, 1500, 2000, 3000, np.inf],
        "labels": ["≤500", "501–1000", "1001–1500", "1501–2000", "2001–3000", ">3000"],
        "titulo": "Distancia del vuelo"
    },
    "Departure Delay in Minutes": {
        "bins": [-np.inf, 0, 15, 30, 60, 120, np.inf],
        "labels": ["0", "1–15", "16–30", "31–60", "61–120", ">120"],
        "titulo": "Retraso de salida (min)"
    },
    "Arrival Delay in Minutes": {
        "bins": [-np.inf, 0, 15, 30, 60, 120, np.inf],
        "labels": ["0", "1–15", "16–30", "31–60", "61–120", ">120"],
        "titulo": "Retraso de llegada (min)"
    }
}

fig, ejes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (variable, config) in zip(ejes.flat, variables_intervalos.items()):
    grupos = pd.cut(train[variable], bins=config["bins"], labels=config["labels"])
    resumen = train.groupby(grupos, observed=True)["satisfaction"].agg(
        tasa="mean", pasajeros="size"
    )
    resumen["tasa"] *= 100

    posiciones = np.arange(len(resumen))
    ax.plot(posiciones, resumen["tasa"], marker="o", lw=2.2, color="#287D8E")

    for posicion, valor in zip(posiciones, resumen["tasa"]):
        ax.annotate(f"{valor:.1f}%", (posicion, valor), xytext=(0, 8),
                    textcoords="offset points", ha="center", fontsize=9)

    etiquetas = [f"{tramo}\n(n={cantidad:,})" for tramo, cantidad
                 in zip(resumen.index, resumen["pasajeros"])]
    ax.set_xticks(posiciones, labels=etiquetas, fontsize=8)
    ax.set_title(config["titulo"], fontsize=13, weight="bold", pad=12)
    ax.set_xlabel("Intervalos")
    ax.set_ylabel("Pasajeros satisfechos (%)")
    ax.set_ylim(0, 105)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Satisfacción según variables numéricas", fontsize=17, weight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(carpeta_imagenes / "numeric_satisfaction_intervals.png", dpi=220, bbox_inches="tight")
plt.show()


**Lectura:** la relación entre satisfacción y edad no parece lineal: las tasas suben en ciertos grupos intermedios y disminuyen entre los mayores. También se aprecian diferencias por distancia y, en general, una menor satisfacción cuando aumentan los retrasos.

Estas diferencias pueden estar relacionadas con otras variables, como clase y motivo del viaje; no debemos interpretarlas como efectos causales. Además, los grupos con retrasos elevados suelen tener menos observaciones que los de retraso cero.

### 7.2. Diagramas de dispersión

Como `satisfaction` es binaria (0/1), los puntos se superponen en dos franjas horizontales. Usamos una pequeña dispersión vertical (*jitter*) **solo para visualizar una muestra**; la línea representa la tasa por intervalos calculada con **todos los datos disponibles**. No modificamos la variable objetivo ni alteramos los registros originales.


In [ ]:
variables_scatter = [
    "Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"
]
muestra = train.sample(n=min(8_000, len(train)), random_state=42)
rng = np.random.default_rng(42)

fig, ejes = plt.subplots(2, 2, figsize=(15, 9))
for ax, variable in zip(ejes.flat, variables_scatter):
    datos_validos = train[[variable, "satisfaction"]].dropna()

    for estado, color, etiqueta in [
        (False, "#D95F5F", "Insatisfecho"),
        (True, "#287D8E", "Satisfecho")
    ]:
        grupo = muestra.loc[muestra["satisfaction"].eq(estado), [variable]].dropna()
        y = int(estado) + rng.uniform(-0.08, 0.08, size=len(grupo))
        ax.scatter(grupo[variable], y, s=8, alpha=0.12,
                   color=color, label=etiqueta, rasterized=True)

    if variable in ("Age", "Flight Distance"):
        tramos = pd.qcut(datos_validos[variable], q=15, duplicates="drop")
    else:
        tramos = pd.cut(
            datos_validos[variable],
            bins=[-np.inf, 0, 5, 15, 30, 60, 120, 300, np.inf]
        )

    tendencia = (
        datos_validos.assign(tramo=tramos)
        .groupby("tramo", observed=True)
        .agg(x=(variable, "median"), tasa=("satisfaction", "mean"), n=("satisfaction", "size"))
    )
    tendencia = tendencia.loc[tendencia["n"] >= 50].sort_values("x")
    ax.plot(tendencia["x"], tendencia["tasa"], "-o", color="#1D2630",
            lw=2, ms=3.5, label="Tasa observada por tramo")

    if "Delay" in variable:
        ax.set_xlim(0, datos_validos[variable].quantile(0.995))
        ax.text(0.99, 0.02, "Eje X hasta P99,5", transform=ax.transAxes,
                ha="right", fontsize=8, color="dimgray")
    ax.set_ylim(-0.16, 1.16)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_title(variable, weight="bold", fontsize=11)
    ax.set_xlabel(variable)
    ax.set_ylabel("Satisfacción (0/1) / proporción")
    ax.grid(alpha=0.18)
    ax.spines[["top", "right"]].set_visible(False)

handles, labels = ejes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3,
           bbox_to_anchor=(0.5, 0.95), frameon=False)
fig.suptitle("Dispersión de variables numéricas y satisfacción",
             fontsize=16, fontweight="bold", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(carpeta_imagenes / "numerical_scatter_satisfaction.png",
            dpi=220, bbox_inches="tight")
plt.show()


## 8. Conclusiones del análisis exploratorio

| Hallazgo | ¿Qué aporta al proyecto? |
|---|---|
| **44,36 %** de pasajeros satisfechos | Conviene mantener las proporciones de las clases al separar entrenamiento y validación. |
| **292 valores faltantes** en `Arrival Delay in Minutes` | Será necesario imputarlos durante el preprocesamiento, sin utilizar datos de validación para ajustar la transformación. |
| Diferencias importantes según `Class` y `Type of Travel` | Vale la pena conservar ambas variables como posibles predictores. |
| La relación entre clase y satisfacción cambia según el motivo del viaje | Sugiere patrones conjuntos que un modelo no lineal, como Random Forest, podría aprovechar. |
| `Online boarding` presenta la asociación de Spearman más alta con el objetivo | Es una variable interesante para el modelado; **correlación no significa importancia predictiva ni causalidad**. |
| Algunas valoraciones iguales a `0` se comportan de forma particular | Debemos interpretar estos códigos con cautela, sin eliminarlos automáticamente. |
| Edad, distancia y retrasos muestran patrones que no siempre son lineales | Tiene sentido comparar un modelo lineal con alternativas capaces de representar relaciones más complejas. |

### Limitaciones

Este análisis se realizó únicamente con **`train.csv`**. Las diferencias observadas describen esta muestra, no efectos causales. Las correlaciones solo incluyen variables numéricas y ordinales; las categorías de texto se estudiaron mediante tablas. Además, al tratarse de una competencia, no podemos asumir que la muestra representa a todos los pasajeros de aerolíneas.

### Continuación del proyecto

En [`02_preprocessing.ipynb`](02_preprocessing.ipynb) preparamos las variables y separamos entrenamiento y validación. En [`03_modelado.ipynb`](03_modelado.ipynb) comparamos regresión logística y Random Forest, utilizamos validación cruzada y optimizamos hiperparámetros. Finalmente, [`04_submission.ipynb`](04_submission.ipynb) entrena el modelo seleccionado con todos los registros etiquetados y genera las probabilidades para Kaggle.

> **Para reproducir este notebook:** descarga `train.csv` de Kaggle, guárdalo en `data/` y ejecuta todas las celdas con un entorno de Python que tenga instalados `numpy`, `pandas`, `matplotlib` y `seaborn`. El notebook crea automáticamente la carpeta `images/` y actualiza las figuras que utiliza el proyecto.
